# Apriori — Non-Member (Guest) Association Rules

Mencari association rules khusus untuk transaksi **Guest / Non-Member**.

**Pipeline:**
1. Load basket data & filter guest transactions
2. Join segment labels dari K-Means guest
3. Run Apriori → frequent itemsets
4. Generate association rules (lift, confidence)
5. Simpan hasil untuk app

## Import Library

In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow.compute as pc
import gc

from mlxtend.frequent_patterns import apriori, association_rules

import warnings
warnings.filterwarnings('ignore')

print('Library imported.')

Library imported.


## Load Basket Data

Menggunakan PyArrow langsung untuk menghindari error categorical dtype.

In [2]:
gc.collect()

# Read basket via PyArrow
basket_table = pq.read_table('df_basket_apriori.parquet')
print(f'Basket table: {basket_table.num_rows:,} rows x {basket_table.num_columns} cols')

gc.collect()

Basket table: 9,064,669 rows x 9 cols


0

## Join dengan Member Status

Ambil `member_status` dari `df_transaction_features.parquet` untuk filter transaksi guest saja.

In [3]:
gc.collect()

# Read transaction features (hanya kolom yang diperlukan)
tx_table = pq.read_table(
    'df_transaction_features.parquet',
    columns=['transaction_id', 'member_status']
)
print(f'TX features: {tx_table.num_rows:,} rows')

gc.collect()

TX features: 14,623,691 rows


0

### Join Basket + Transaction Features via PyArrow

In [4]:
gc.collect()

# Join basket dengan member_status
join_table = basket_table.join(
    tx_table,
    keys='transaction_id',
    join_type='inner'
)

print(f'Joined table: {join_table.num_rows:,} rows')
gc.collect()

# Filter hanya GUEST
mask_guest = pc.equal(join_table.column('member_status'), 'Guest')
guest_table = join_table.filter(mask_guest)
print(f'Guest transactions: {guest_table.num_rows:,} rows')

# Drop member_status column
guest_table = guest_table.drop(['member_status'])

gc.collect()

Joined table: 9,064,669 rows
Guest transactions: 4,532,074 rows


0

### Konversi ke Pandas DataFrame (untuk mlxtend)

In [5]:
gc.collect()

# Konversi ke pandas
guest_pd = guest_table.to_pandas()

# Set transaction_id sebagai index
guest_pd = guest_pd.set_index('transaction_id')

# Cast ke bool (0/1)
guest_pd = guest_pd.astype(bool)

print(f'Guest basket shape: {guest_pd.shape}')
print(f'Guest unique transactions: {guest_pd.index.nunique():,}')
print()
print('Sample (5 transaksi pertama):')
print(guest_pd.head(5))

del join_table, tx_table, basket_table
gc.collect()

Guest basket shape: (4532074, 8)
Guest unique transactions: 4,532,074

Sample (5 transaksi pertama):
                                      Americano  Cappuccino  Espresso  \
transaction_id                                                          
1e89bc3b-364c-4da9-afb6-6e2849ed096c      False       False     False   
1e89bf54-ab42-4e67-863a-90398cfcf39e      False       False     False   
1e89c048-ec9e-454c-a066-9a19d059c61b      False       False     False   
1e89c0bc-28d5-4cf9-a539-432db4583dd4       True       False     False   
1e89c264-d7c2-4026-9c7f-d74c3d87d436      False        True     False   

                                      Flat White  Hot Chocolate  Latte  \
transaction_id                                                           
1e89bc3b-364c-4da9-afb6-6e2849ed096c       False          False   True   
1e89bf54-ab42-4e67-863a-90398cfcf39e       False           True  False   
1e89c048-ec9e-454c-a066-9a19d059c61b       False           True   True   
1e89c0bc-28d5-4cf

0

## Run Apriori — Frequent Itemsets

Mencari itemset yang sering muncul bersama dalam transaksi guest.

In [6]:
min_support = 0.01
print(f'Running Apriori with min_support={min_support}...')
gc.collect()

frequent_itemsets = apriori(
    guest_pd,
    min_support=min_support,
    use_colnames=True,
    low_memory=True
)

print(f'Frequent itemsets found: {len(frequent_itemsets):,}')
print(frequent_itemsets.head(10))

Running Apriori with min_support=0.01...
Frequent itemsets found: 36
    support                 itemsets
0  0.294496              (Americano)
1  0.294118             (Cappuccino)
2  0.294274               (Espresso)
3  0.293829             (Flat White)
4  0.294347          (Hot Chocolate)
5  0.294215                  (Latte)
6  0.293536           (Matcha Latte)
7  0.293926                  (Mocha)
8  0.061027  (Americano, Cappuccino)
9  0.061153    (Americano, Espresso)


## Generate Association Rules

Filter berdasarkan lift > 1 (positive association) dan confidence minimal.

In [7]:
min_threshold = 0
print(f'Generating association rules with min_threshold={min_threshold}...')
gc.collect()

rules = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=min_threshold
)

# Sort by lift descending
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Association rules found: {len(rules):,}')
print()
print('Top 10 rules by lift:')
display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
print(rules[display_cols].head(10))

Generating association rules with min_threshold=1.0...
Association rules found: 0

Top 10 rules by lift:
Empty DataFrame
Columns: [antecedents, consequents, support, confidence, lift]
Index: []


In [8]:
# Summary of rules
print('=== Rules Summary ===')
print(f'Total rules: {len(rules):,}')
print(f'Avg lift: {rules["lift"].mean():.3f}')
print(f'Avg confidence: {rules["confidence"].mean():.3f}')
print(f'Avg support: {rules["support"].mean():.4f}')
print()
print(f'Rules with lift > 2: {len(rules[rules["lift"] > 2]):,}')
print(f'Rules with lift > 3: {len(rules[rules["lift"] > 3]):,}')
print(f'Rules with lift > 5: {len(rules[rules["lift"] > 5]):,}')

=== Rules Summary ===
Total rules: 0
Avg lift: nan
Avg confidence: nan
Avg support: nan

Rules with lift > 2: 0
Rules with lift > 3: 0
Rules with lift > 5: 0


## Save Results

Menyimpan rules untuk digunakan di app.

In [9]:
gc.collect()

# Konversi frozenset ke string agar bisa disimpan
rules_out = rules.copy()
rules_out['antecedents'] = rules_out['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
rules_out['consequents'] = rules_out['consequents'].apply(lambda x: ', '.join(sorted(list(x))))

# Save parquet
rules_out.to_parquet('df_rules_guest.parquet', index=False)
print('Saved: df_rules_guest.parquet')

# Save CSV (view-friendly)
rules_out.to_csv('df_rules_guest.csv', index=False)
print('Saved: df_rules_guest.csv')

print(f'\nDone. {len(rules_out):,} association rules saved.')

Saved: df_rules_guest.parquet
Saved: df_rules_guest.csv

Done. 0 association rules saved.
